# Feature Engineering - GUIDE Dataset (No Data Leakage)

**Obiettivo:** Feature engineering avanzato con target binario e encoding ottimizzato.

**Caratteristiche Principali:**
- Target binario: BinaryIncidentGrade (1=TruePositive, 0=FalsePositive/BenignPositive)
- SmoothedRisk per AlertTitle (Bayesian smoothing) - **calcolato solo su train**
- GeoLoc_freq - **calcolato solo su train**
- Frequency encoding per colonne ad alta cardinalità - **calcolato solo su train**
- MITRE top 30 tecniche
- One-hot encoding selettivo solo per SuspicionLevel e EvidenceRole

**Pipeline (Corretta per evitare Data Leakage):**
1. Caricamento e pulizia
2. Target binario
3. **Train/Test split a livello IncidentId (PRIMA del feature engineering)**
4. SmoothedRisk per AlertTitle (solo su train, applicato a test)
5. GeoLoc_freq (solo su train, applicato a test)
6. Features temporali
7. Frequency encoding categorie (solo su train, applicato a test)
8. One-hot encoding selettivo
9. Processing MITRE (top 30 da train)
10. Aggregazione Evidence → Incident
11. Salvataggio

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

print("Librerie importate con successo!")

Librerie importate con successo!


## 2. Caricamento e Pulizia

In [2]:
print("Caricamento dataset...")
df = pd.read_csv('../data/GUIDE_Train.csv')

print(f"Dataset caricato: {df.shape[0]:,} righe, {df.shape[1]} colonne")

# Rimuovi record senza target
df = df[df['IncidentGrade'].notna()].copy()

# Rimuovi duplicati
#df = df.drop_duplicates(subset=['Id'], keep='first')

print(f"Dimensioni dopo pulizia: {df.shape}")

Caricamento dataset...
Dataset caricato: 9,516,837 righe, 45 colonne
Dimensioni dopo pulizia: (9465497, 45)


## 3. Target Binario

In [3]:
# Crea target binario: 1 = TruePositive, 0 = FalsePositive/BenignPositive
df['BinaryIncidentGrade'] = df['IncidentGrade'].apply(
    lambda x: 1 if x == 'TruePositive' else 0
)

counts = df['BinaryIncidentGrade'].value_counts()
percentages = df['BinaryIncidentGrade'].value_counts(normalize=True).mul(100).round(2)

result = pd.DataFrame({'Count': counts, 'Percentage': percentages})
print("\nDistribuzione Target Binario:")
print(result)
print(f"\nClass imbalance ratio: {counts[0]/counts[1]:.2f}:1")


Distribuzione Target Binario:
                       Count  Percentage
BinaryIncidentGrade                     
0                    6142784        64.9
1                    3322713        35.1

Class imbalance ratio: 1.85:1


## 4. Train/Test Split a Livello IncidentId (PRIMA del Feature Engineering)

**CRITICO:** Lo split deve avvenire PRIMA di calcolare qualsiasi statistica (SmoothedRisk, frequenze, ecc.) per evitare data leakage. Splittiamo a livello di IncidentId, poi separiamo le evidenze.

In [4]:
# Ottieni IncidentId univoci con il loro target
incident_targets = df.groupby('IncidentId')['BinaryIncidentGrade'].first().reset_index()

# Split stratificato a livello IncidentId (70/30)
train_incidents, test_incidents = train_test_split(
    incident_targets['IncidentId'],
    test_size=0.3,
    stratify=incident_targets['BinaryIncidentGrade'],
    random_state=42
)

# Separa le evidenze in train e test
df_train = df[df['IncidentId'].isin(train_incidents)].copy()
df_test = df[df['IncidentId'].isin(test_incidents)].copy()

print(f"Split a livello IncidentId completato:")
print(f"  Train incidents: {len(train_incidents):,} ({len(train_incidents)/len(incident_targets)*100:.1f}%)")
print(f"  Test incidents: {len(test_incidents):,} ({len(test_incidents)/len(incident_targets)*100:.1f}%)")
print(f"\n  Train evidences: {len(df_train):,}")
print(f"  Test evidences: {len(df_test):,}")

# Verifica stratificazione
print(f"\nDistribuzione target (train):")
print(df_train.groupby('IncidentId')['BinaryIncidentGrade'].first().value_counts(normalize=True))
print(f"\nDistribuzione target (test):")
print(df_test.groupby('IncidentId')['BinaryIncidentGrade'].first().value_counts(normalize=True))

Split a livello IncidentId completato:
  Train incidents: 314,230 (70.0%)
  Test incidents: 134,671 (30.0%)

  Train evidences: 6,642,509
  Test evidences: 2,822,988

Distribuzione target (train):
BinaryIncidentGrade
0    0.78701
1    0.21299
Name: proportion, dtype: float64

Distribuzione target (test):
BinaryIncidentGrade
0    0.787007
1    0.212993
Name: proportion, dtype: float64


## 5. SmoothedRisk per AlertTitle (Solo Train → Applicato a Test)

**Wilson/Bayes Smoothing:** Corregge la media quando abbiamo pochi esempi.  
Con solo 2 esempi e 100% risk, lo smoothing porta il valore verso 0.5 per riflettere l'incertezza.

**⚠️ NO DATA LEAKAGE:** Calcoliamo le statistiche SOLO sul train set, poi applichiamo la mappatura al test.

In [5]:
# Calcola risk grezzo per AlertTitle SOLO SU TRAIN
alert_risk = df_train.groupby('AlertTitle')['BinaryIncidentGrade'].mean()
alert_count = df_train.groupby('AlertTitle')['BinaryIncidentGrade'].count()

alert_summary = pd.DataFrame({
    'Risk': alert_risk,
    'Count': alert_count
})

# Bayesian smoothing
alpha = 2
beta = 2

alert_summary['SmoothedRisk'] = (
    alert_summary['Risk'] * alert_summary['Count'] + alpha
) / (alert_summary['Count'] + alpha + beta)

# Valore di fallback per AlertTitle non visti nel train (prior neutro)
global_prior = alpha / (alpha + beta)  # 0.5

# Applica a TRAIN
df_train = df_train.merge(
    alert_summary[['SmoothedRisk']], 
    left_on='AlertTitle', 
    right_index=True, 
    how='left'
)
df_train['SmoothedRisk'] = df_train['SmoothedRisk'].fillna(global_prior)

# Applica a TEST (con fallback per AlertTitle non visti)
df_test = df_test.merge(
    alert_summary[['SmoothedRisk']], 
    left_on='AlertTitle', 
    right_index=True, 
    how='left'
)
df_test['SmoothedRisk'] = df_test['SmoothedRisk'].fillna(global_prior)

# Statistiche
unseen_alerts = df_test['AlertTitle'].nunique() - df_test[df_test['AlertTitle'].isin(alert_summary.index)]['AlertTitle'].nunique()
print(f"SmoothedRisk calcolato su {len(alert_summary)} AlertTitle univoci (da train)")
print(f"AlertTitle nel test non visti nel train: {unseen_alerts} (useranno prior={global_prior})")
print(f"\nStatistiche SmoothedRisk (train):")
print(df_train['SmoothedRisk'].describe())
print(f"\nStatistiche SmoothedRisk (test):")
print(df_test['SmoothedRisk'].describe())

SmoothedRisk calcolato su 62844 AlertTitle univoci (da train)
AlertTitle nel test non visti nel train: 17108 (useranno prior=0.5)

Statistiche SmoothedRisk (train):
count    6.642509e+06
mean     3.631533e-01
std      3.405518e-01
min      2.002684e-05
25%      3.415702e-02
50%      2.615600e-01
75%      7.573722e-01
max      9.999489e-01
Name: SmoothedRisk, dtype: float64

Statistiche SmoothedRisk (test):
count    2.822988e+06
mean     3.816584e-01
std      3.288454e-01
min      2.002684e-05
25%      6.474820e-02
50%      3.464084e-01
75%      7.573722e-01
max      9.999489e-01
Name: SmoothedRisk, dtype: float64


## 6. GeoLoc_freq (Solo Train → Applicato a Test)

Frequenza normalizzata della combinazione `CountryCode_State_City`.  
Utile per identificare location rare o pattern regionali.

**⚠️ NO DATA LEAKAGE:** Frequenze calcolate SOLO sul train set.

In [6]:
# Crea identificatore geografico
for df_part in [df_train, df_test]:
    df_part['GeoLoc'] = (
        df_part['CountryCode'].astype(str) + "_" + 
        df_part['State'].astype(str) + "_" + 
        df_part['City'].astype(str)
    )

# Calcola frequenza normalizzata SOLO SU TRAIN
geo_freq = df_train['GeoLoc'].value_counts(normalize=True).to_dict()

# Valore di fallback per location non viste (mediana delle frequenze)
geo_fallback = np.median(list(geo_freq.values()))

# Applica a train e test
df_train['GeoLoc_freq'] = df_train['GeoLoc'].map(geo_freq)
df_test['GeoLoc_freq'] = df_test['GeoLoc'].map(geo_freq).fillna(geo_fallback)

# Drop colonne geografiche originali
for df_part in [df_train, df_test]:
    df_part.drop(columns=['CountryCode', 'State', 'City', 'GeoLoc'], inplace=True, errors='ignore')

unseen_geo = df_test['GeoLoc_freq'].isna().sum()  # Prima del fillna
print(f"GeoLoc_freq calcolato su {len(geo_freq)} location univoche (da train)")
print(f"Location nel test non viste: useranno fallback={geo_fallback:.6f}")
print(f"\nStatistiche GeoLoc_freq (train):")
print(df_train['GeoLoc_freq'].describe())

GeoLoc_freq calcolato su 9232 location univoche (da train)
Location nel test non viste: useranno fallback=0.000000

Statistiche GeoLoc_freq (train):
count    6.642509e+06
mean     8.494171e-01
std      2.472238e-01
min      1.505455e-07
25%      9.215489e-01
50%      9.215489e-01
75%      9.215489e-01
max      9.215489e-01
Name: GeoLoc_freq, dtype: float64


## 7. Features Temporali

Le features temporali non causano data leakage (sono estratte direttamente dal timestamp, non calcolate su statistiche del dataset).

In [7]:
for df_part in [df_train, df_test]:
    df_part['Timestamp'] = pd.to_datetime(df_part['Timestamp'])
    df_part['month'] = df_part['Timestamp'].dt.month
    df_part['hour'] = df_part['Timestamp'].dt.hour
    df_part['weekday'] = df_part['Timestamp'].dt.weekday + 1
    df_part['IsWeekend'] = (df_part['Timestamp'].dt.dayofweek >= 5).astype(int)

print("Features temporali create: month, hour, weekday, IsWeekend")

Features temporali create: month, hour, weekday, IsWeekend


## 8. Gestione Missing e Frequency Encoding (Solo Train → Applicato a Test)

**Frequency encoding** per colonne ad alta cardinalità evita curse of dimensionality.  
Fornisce un segnale semplice e limitato.

**⚠️ NO DATA LEAKAGE:** Frequenze calcolate SOLO sul train set.

In [8]:
# Fill missing values (su entrambi i set)
for df_part in [df_train, df_test]:
    df_part['Roles'] = df_part['Roles'].fillna('missing')
    df_part['ActionGrouped'] = df_part['ActionGrouped'].fillna('Missing')
    df_part['SuspicionLevel'] = df_part['SuspicionLevel'].fillna('Missing')
    df_part['LastVerdict'] = df_part['LastVerdict'].fillna('Missing')

# Group rare verdicts (< 100 occorrenze) - basato SOLO su train
verdict_counts = df_train['LastVerdict'].value_counts()
rare_verdicts = verdict_counts[verdict_counts < 100].index.tolist()

for df_part in [df_train, df_test]:
    df_part['LastVerdict'] = df_part['LastVerdict'].replace(rare_verdicts, 'Other')

print("Missing values gestiti")
print(f"Verdicts raggruppati in 'Other': {len(rare_verdicts)}")

Missing values gestiti
Verdicts raggruppati in 'Other': 2


In [9]:
# Frequency encoding per colonne ad alta cardinalità
# CALCOLATO SOLO SU TRAIN, APPLICATO A TEST

freq_encode_cols = [
    'ThreatFamily', 'AntispamDirection', 'ActionGranular',
    'LastVerdict', 'ResourceType', 'Roles', 'ActionGrouped', 
    'EntityType', 'Category'
]

# Dizionario per salvare le mappature (utile per inference futura)
freq_mappings = {}

for col in freq_encode_cols:
    if col in df_train.columns:
        # Fill missing
        df_train[col] = df_train[col].fillna('Missing')
        df_test[col] = df_test[col].fillna('Missing')
        
        # Calcola frequenza SOLO SU TRAIN
        freq = df_train[col].value_counts(normalize=True).to_dict()
        freq_mappings[col] = freq
        
        # Fallback per valori non visti (mediana delle frequenze)
        fallback = np.median(list(freq.values()))
        
        # Applica a train e test
        df_train[f"{col}_freq"] = df_train[col].map(freq)
        df_test[f"{col}_freq"] = df_test[col].map(freq).fillna(fallback)
        
        # Drop original
        df_train.drop(columns=col, inplace=True)
        df_test.drop(columns=col, inplace=True)
        
        unseen = df_test[f"{col}_freq"].isna().sum()
        print(f"  {col} -> {col}_freq (unseen in test: {unseen}, fallback={fallback:.4f})")

print(f"\nFrequency encoding completato per {len(freq_encode_cols)} colonne")

  ThreatFamily -> ThreatFamily_freq (unseen in test: 0, fallback=0.0000)
  AntispamDirection -> AntispamDirection_freq (unseen in test: 0, fallback=0.0009)
  ActionGranular -> ActionGranular_freq (unseen in test: 0, fallback=0.0000)
  LastVerdict -> LastVerdict_freq (unseen in test: 0, fallback=0.0466)
  ResourceType -> ResourceType_freq (unseen in test: 0, fallback=0.0000)
  Roles -> Roles_freq (unseen in test: 0, fallback=0.0014)
  ActionGrouped -> ActionGrouped_freq (unseen in test: 0, fallback=0.0003)
  EntityType -> EntityType_freq (unseen in test: 0, fallback=0.0003)
  Category -> Category_freq (unseen in test: 0, fallback=0.0066)

Frequency encoding completato per 9 colonne


## 9. One-Hot Encoding Selettivo

Solo per `SuspicionLevel` e `EvidenceRole` (bassa cardinalità e forte segnale).

**Nota:** Le categorie rare vengono raggruppate basandosi SOLO sui conteggi del train set.

In [10]:
onehot_cols = ['SuspicionLevel', 'EvidenceRole']

for col in onehot_cols:
    if col in df_train.columns:
        # Fill missing
        df_train[col] = df_train[col].fillna('Missing')
        df_test[col] = df_test[col].fillna('Missing')
        
        # Group rare categories (< 100 occorrenze) - basato SOLO su train
        counts = df_train[col].value_counts()
        rare = counts[counts < 100].index.tolist()
        
        df_train[col] = df_train[col].replace(rare, 'Other')
        df_test[col] = df_test[col].replace(rare, 'Other')
        
        # Ottieni le categorie valide dal train
        train_categories = df_train[col].unique().tolist()
        
        # Mappa categorie non viste nel test a 'Other'
        df_test[col] = df_test[col].apply(
            lambda x: x if x in train_categories else 'Other'
        )
        
        print(f"  {col}: categorie valide = {train_categories}")

# One-hot encode (stesso set di colonne per train e test)
# Usiamo pd.get_dummies con le stesse categorie
df_train = pd.get_dummies(df_train, columns=['SuspicionLevel', 'EvidenceRole'], drop_first=True)
df_test = pd.get_dummies(df_test, columns=['SuspicionLevel', 'EvidenceRole'], drop_first=True)

# Allinea le colonne (aggiungi colonne mancanti nel test, rimuovi extra)
train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

# Aggiungi colonne mancanti nel test (con valore 0)
for col in train_cols - test_cols:
    df_test[col] = 0

# Rimuovi colonne extra nel test
for col in test_cols - train_cols:
    if col.startswith(('SuspicionLevel_', 'EvidenceRole_')):
        df_test.drop(columns=col, inplace=True)

print("\nOne-hot encoding completato")
print(f"Colonne train: {df_train.shape[1]}, Colonne test: {df_test.shape[1]}")

  SuspicionLevel: categorie valide = ['Missing', 'Suspicious', 'Incriminated']
  EvidenceRole: categorie valide = ['Impacted', 'Related']

One-hot encoding completato
Colonne train: 50, Colonne test: 50


## 10. Processing MITRE Techniques (Top 30 da Train)

**⚠️ NO DATA LEAKAGE:** Le top 30 tecniche vengono selezionate SOLO dal train set.

In [11]:
# Step 1: Split semicolon-separated string (su entrambi)
for df_part in [df_train, df_test]:
    df_part['MitreList'] = df_part['MitreTechniques'].apply(
        lambda x: x.split(';') if pd.notna(x) else []
    )

# Step 2: Identifica top 30 tecniche SOLO DA TRAIN
all_techs_train = [tech for sublist in df_train['MitreList'] for tech in sublist]
top_techs = [tech for tech, _ in Counter(all_techs_train).most_common(30)]
top_tech_set = set(top_techs)

print(f"Top 30 MITRE techniques selezionate (da train)")
print(f"Top 10: {Counter(all_techs_train).most_common(10)}")

Top 30 MITRE techniques selezionate (da train)
Top 10: [('T1078', 1030969), ('T1078.004', 953625), ('T1566.002', 577553), ('T1566', 477409), ('T1110', 124793), ('T1133', 121320), ('T1566.001', 97838), ('T1110.003', 73190), ('T1110.001', 72351), ('T1071', 68076)]


In [12]:
# Step 3: Filtra ogni lista per includere solo top techniques (su entrambi)
for df_part in [df_train, df_test]:
    df_part['FilteredMitreList'] = df_part['MitreList'].apply(
        lambda x: [tech for tech in x if tech in top_tech_set]
    )

# Step 4: One-hot encode con MultiLabelBinarizer (stesso set di classi per train e test)
mlb = MultiLabelBinarizer(classes=top_techs)

# Applica a train
tech_matrix_train = pd.DataFrame(
    mlb.fit_transform(df_train['FilteredMitreList']),
    columns=mlb.classes_, 
    index=df_train.index
)

# Applica a test (usa transform, non fit_transform)
tech_matrix_test = pd.DataFrame(
    mlb.transform(df_test['FilteredMitreList']),
    columns=mlb.classes_, 
    index=df_test.index
)

# Step 5: Merge e drop colonne originali
df_train = pd.concat([df_train, tech_matrix_train], axis=1)
df_test = pd.concat([df_test, tech_matrix_test], axis=1)

for df_part in [df_train, df_test]:
    df_part.drop(columns=['MitreTechniques', 'MitreList', 'FilteredMitreList'], inplace=True)

print(f"\nMITRE features create: {tech_matrix_train.shape[1]}")
print(f"Shape train: {df_train.shape}")
print(f"Shape test: {df_test.shape}")


MITRE features create: 30
Shape train: (6642509, 79)
Shape test: (2822988, 79)


## 11. Aggregazione a Livello Incident

Aggreghiamo separatamente train e test per mantenere la separazione.

In [13]:
def get_mode(x):
    mode = x.mode()
    return mode[0] if len(mode) > 0 else x.iloc[0] if len(x) > 0 else None

def build_agg_dict(df_sample):
    """Costruisce il dizionario di aggregazione basato sulle colonne presenti"""
    agg_dict = {
        'BinaryIncidentGrade': 'first',
        'IncidentGrade': 'first',
        'AlertId': 'nunique',
        'Id': 'count',
        'SmoothedRisk': 'mean',
        'GeoLoc_freq': 'mean',
        'hour': ['min', 'max', 'mean'],
        'month': get_mode,
        'weekday': get_mode,
        'IsWeekend': 'max',
        'Timestamp': ['min', 'max'],
    }
    
    # Aggiungi frequency-encoded columns (media)
    freq_cols = [col for col in df_sample.columns if col.endswith('_freq') and col not in ['GeoLoc_freq']]
    for col in freq_cols:
        agg_dict[col] = 'mean'
    
    # Aggiungi one-hot encoded columns (somma)
    onehot_cols_created = [col for col in df_sample.columns if col.startswith(('SuspicionLevel_', 'EvidenceRole_'))]
    for col in onehot_cols_created:
        agg_dict[col] = 'sum'
    
    # Aggiungi MITRE columns (somma) - usa regex per sicurezza
    import re
    mitre_cols = [col for col in df_sample.columns if re.match(r'^T\d{4}$', col)]
    for col in mitre_cols:
        agg_dict[col] = 'sum'
    
    return agg_dict

agg_dict = build_agg_dict(df_train)
print(f"Aggregazioni preparate per {len(agg_dict)} features")

Aggregazioni preparate per 43 features


In [14]:
# Esegui aggregazione su TRAIN
incident_train = df_train.groupby('IncidentId').agg(agg_dict).reset_index()

# Esegui aggregazione su TEST
incident_test = df_test.groupby('IncidentId').agg(agg_dict).reset_index()

# Flatten colonne multi-livello
for df_agg in [incident_train, incident_test]:
    df_agg.columns = [
        '_'.join(col).strip('_') if isinstance(col, tuple) else col 
        for col in df_agg.columns.values
    ]

print(f"Dataset train aggregato: {incident_train.shape}")
print(f"Dataset test aggregato: {incident_test.shape}")

Dataset train aggregato: (314230, 47)
Dataset test aggregato: (134671, 47)


In [15]:
# Calcola durata e rinomina colonne
rename_map = {
    'AlertId_nunique': 'NumAlerts',
    'Id_count': 'NumEvidences',
    'SmoothedRisk_mean': 'SmoothedRisk_avg',
    'GeoLoc_freq_mean': 'GeoLoc_freq_avg',
    'hour_min': 'Hour_First',
    'hour_max': 'Hour_Last',
    'hour_mean': 'Hour_Avg',
    'BinaryIncidentGrade_first': 'BinaryIncidentGrade',
    'IncidentGrade_first': 'IncidentGrade',
}

for df_agg in [incident_train, incident_test]:
    df_agg['Duration_seconds'] = (
        pd.to_datetime(df_agg['Timestamp_max']) - 
        pd.to_datetime(df_agg['Timestamp_min'])
    ).dt.total_seconds()
    
    df_agg.rename(columns=rename_map, inplace=True)
    df_agg.drop(columns=['Timestamp_min', 'Timestamp_max'], errors='ignore', inplace=True)

print(f"Features finali train: {incident_train.shape[1] - 3}")  # -3 per ID e 2 target
print(f"Features finali test: {incident_test.shape[1] - 3}")

Features finali train: 43
Features finali test: 43


## 12. Preparazione per Modeling

In [16]:
# Separa features e target
X_train = incident_train.drop(columns=['IncidentId', 'BinaryIncidentGrade', 'IncidentGrade'])
y_train = incident_train['BinaryIncidentGrade']

X_test = incident_test.drop(columns=['IncidentId', 'BinaryIncidentGrade', 'IncidentGrade'])
y_test = incident_test['BinaryIncidentGrade']

# Assicura che train e test abbiano le stesse colonne nello stesso ordine
common_cols = list(X_train.columns)
X_test = X_test[common_cols]

# Gestisci missing (se presenti)
X_train = X_train.fillna(-999)
X_test = X_test.fillna(-999)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"\nDistribuzione y_train:")
print(y_train.value_counts(normalize=True))
print(f"\nDistribuzione y_test:")
print(y_test.value_counts(normalize=True))

X_train: (314230, 43)
X_test: (134671, 43)

Distribuzione y_train:
BinaryIncidentGrade
0    0.78701
1    0.21299
Name: proportion, dtype: float64

Distribuzione y_test:
BinaryIncidentGrade
0    0.787007
1    0.212993
Name: proportion, dtype: float64


## 13. Salvataggio Dataset Processati

In [17]:
# Salva in una nuova cartella per distinguere dalla versione con data leakage
os.makedirs('../data/processed_v4_noleakage', exist_ok=True)

X_train.to_csv('../data/processed_v4_noleakage/X_train.csv', index=False)
X_test.to_csv('../data/processed_v4_noleakage/X_test.csv', index=False)
y_train.to_csv('../data/processed_v4_noleakage/y_train.csv', index=False, header=['BinaryIncidentGrade'])
y_test.to_csv('../data/processed_v4_noleakage/y_test.csv', index=False, header=['BinaryIncidentGrade'])

# Salva anche i dataframe completi aggregati
incident_train.to_csv('../data/processed_v4_noleakage/incident_train.csv', index=False)
incident_test.to_csv('../data/processed_v4_noleakage/incident_test.csv', index=False)

# Salva le mappature per inference futura
mappings = {
    'alert_smoothed_risk': alert_summary['SmoothedRisk'].to_dict(),
    'geo_freq': geo_freq,
    'freq_encodings': freq_mappings,
    'top_mitre_techniques': top_techs,
    'global_prior': global_prior,
    'geo_fallback': geo_fallback,
}

with open('../data/processed_v4_noleakage/feature_mappings.pkl', 'wb') as f:
    pickle.dump(mappings, f)

print("Dataset salvati in ../data/processed_v4_noleakage/")
print(f"  - X_train.csv: {X_train.shape}")
print(f"  - X_test.csv: {X_test.shape}")
print(f"  - y_train.csv: {y_train.shape}")
print(f"  - y_test.csv: {y_test.shape}")
print(f"  - incident_train.csv: {incident_train.shape}")
print(f"  - incident_test.csv: {incident_test.shape}")
print(f"  - feature_mappings.pkl: mappature per inference")
print(f"\nFeatures totali: {X_train.shape[1]}")
print(f"Target binario: 0 (FP/BP) vs 1 (TP)")
print(f"\n✅ NO DATA LEAKAGE: tutte le statistiche calcolate solo su train!")

Dataset salvati in ../data/processed_v4_noleakage/
  - X_train.csv: (314230, 43)
  - X_test.csv: (134671, 43)
  - y_train.csv: (314230,)
  - y_test.csv: (134671,)
  - incident_train.csv: (314230, 46)
  - incident_test.csv: (134671, 46)
  - feature_mappings.pkl: mappature per inference

Features totali: 43
Target binario: 0 (FP/BP) vs 1 (TP)

✅ NO DATA LEAKAGE: tutte le statistiche calcolate solo su train!


## Summary

**Features create:**
- Target binario (BinaryIncidentGrade)
- SmoothedRisk (Bayesian smoothing per AlertTitle) - **calcolato solo su train**
- GeoLoc_freq (frequenza location) - **calcolato solo su train**
- Temporal features (hour, month, weekday, IsWeekend, Duration_seconds)
- Frequency encoding per 9 colonne ad alta cardinalità - **calcolato solo su train**
- One-hot encoding per SuspicionLevel e EvidenceRole
- MITRE top 30 tecniche (one-hot encoded) - **selezionate solo da train**
- Aggregazioni a livello incident (count, mean, sum, mode)

**✅ DATA LEAKAGE ELIMINATO:**
- Split a livello IncidentId PRIMA del feature engineering
- Tutte le statistiche (frequenze, smoothed risk) calcolate SOLO su train
- Mappature salvate per applicazione su nuovi dati (inference)

**Pronto per training XGBoost con target binario!**